In [7]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

TE = np.arange(0, 305, 5)

# Valeurs de référence par défaut
DEF_MWF = 20
DEF_T2MW = 20
DEF_T2IEW = 100

# Plages de variation pour chaque curseur indépendant
mwf_list = [5, 10, 15, 20, 30, 40]
t2mw_list = [10, 15, 20, 30, 40]
t2iew_list = [60, 80, 100, 140, 180]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Biexponential T2 signal decay", "Relative myelin water contribution")
)

colors = ["#1f77b4", "#2ca02c", "#111111", "#d62728"]

def add_state_traces(mwf, t2mw, t2iew, visible=False):
    A_mw = mwf / 100.0
    A_iew = 1.0 - A_mw
    S_mw = A_mw * np.exp(-TE / t2mw)
    S_iew = A_iew * np.exp(-TE / t2iew)
    S_total = S_mw + S_iew
    frac_mw = 100.0 * S_mw / S_total

    fig.add_trace(go.Scatter(x=TE, y=S_mw, name="Myelin water (MW)", line=dict(color=colors[0]), visible=visible), row=1, col=1)
    fig.add_trace(go.Scatter(x=TE, y=S_iew, name="Intra/extracellular (IEW)", line=dict(color=colors[1]), visible=visible), row=1, col=1)
    fig.add_trace(go.Scatter(x=TE, y=S_total, name="Total signal", line=dict(color=colors[2], width=2.5), visible=visible), row=1, col=1)
    fig.add_trace(go.Scatter(x=TE, y=frac_mw, name="MW contribution (%)", line=dict(color=colors[3], width=2.5), visible=visible, showlegend=False), row=1, col=2)

# Construire la liste des états (seulement 16 états au total !)
states = []
# Groupe 1 : variation de MWF
for m in mwf_list:
    is_def = (m == DEF_MWF)
    add_state_traces(m, DEF_T2MW, DEF_T2IEW, visible=is_def)
    states.append((m, DEF_T2MW, DEF_T2IEW))

# Groupe 2 : variation de T2_MW
for t_mw in t2mw_list:
    add_state_traces(DEF_MWF, t_mw, DEF_T2IEW, visible=False)
    states.append((DEF_MWF, t_mw, DEF_T2IEW))

# Groupe 3 : variation de T2_IEW
for t_iew in t2iew_list:
    add_state_traces(DEF_MWF, DEF_T2MW, t_iew, visible=False)
    states.append((DEF_MWF, DEF_T2MW, t_iew))

total_traces = len(fig.data)

def make_steps(start_idx, count, label_fmt):
    steps = []
    for i in range(count):
        state_idx = start_idx + i
        m, tmw, tiew = states[state_idx]
        vis = [False] * total_traces
        for k in range(4):
            vis[4 * state_idx + k] = True
        steps.append(
            dict(
                method="update",
                args=[
                    {"visible": vis},
                    {"title": f"MWF = {m}% | T2,MW = {tmw} ms | T2,IEW = {tiew} ms"}
                ],
                label=label_fmt(m, tmw, tiew)
            )
        )
    return steps

steps_mwf = make_steps(0, len(mwf_list), lambda m, tmw, tiew: f"{m}%")
steps_t2mw = make_steps(len(mwf_list), len(t2mw_list), lambda m, tmw, tiew: f"{tmw} ms")
steps_t2iew = make_steps(len(mwf_list) + len(t2mw_list), len(t2iew_list), lambda m, tmw, tiew: f"{tiew} ms")

fig.update_layout(
    title=f"MWF = {DEF_MWF}% | T2,MW = {DEF_T2MW} ms | T2,IEW = {DEF_T2IEW} ms",
    template="plotly_white",
    height=680,
    margin=dict(b=240),
    sliders=[
        dict(
            active=mwf_list.index(DEF_MWF),
            currentvalue={"prefix": "1) Vary MWF (T2 fixed): "},
            y=-0.12,
            pad={"t": 10},
            steps=steps_mwf
        ),
        dict(
            active=t2mw_list.index(DEF_T2MW),
            currentvalue={"prefix": "2) Vary T2,MW (MWF=20%, T2,IEW=100ms): "},
            y=-0.32,
            pad={"t": 10},
            steps=steps_t2mw
        ),
        dict(
            active=t2iew_list.index(DEF_T2IEW),
            currentvalue={"prefix": "3) Vary T2,IEW (MWF=20%, T2,MW=20ms): "},
            y=-0.52,
            pad={"t": 10},
            steps=steps_t2iew
        ),
    ]
)

fig.update_xaxes(title_text="Echo time TE (ms)", row=1, col=1)
fig.update_xaxes(title_text="Echo time TE (ms)", row=1, col=2)
fig.update_yaxes(title_text="Normalized signal", range=[0, 1.05], row=1, col=1)
fig.update_yaxes(title_text="MW contribution (%)", range=[0, 50], row=1, col=2)

fig.show()